## Overview
It is a protocol to connect AI applications to external systems. For example, Claude or ChatGPT can connect to databases, private wikis, tools like search engines, etc.

![MCP](./images/mcp_overview.png)

## Architecture
Participants are:
- **MCP Host:** The AI application that coordinates and manages one or multiple MCP clients
- **MCP Client:** A component that maintains a connection to an MCP server and obtains context from an MCP server for the MCP host to use
- **MCP Server:** A program that provides context to MCP clients

A MCP host creates one or more MCP clients to interact with the MCP server.

![MCP](./images/participants.png)

MCP Server can run locally or it can be hosted on a remote server.

## Data Layer Protocol
MCP is stateful and requires lifecycle management. Client and server both negotiate the capabilities that each one posseses. These capabilities can be listed in terms of primitives:

### Primitives
Define what the client and server can offer each other. There are three primary primitives in the MCP specification:
1. **Resources:** are read only data sources. They allow the AI model to pull in context from files, databases, or APIs. They function similarly to a GET request in web development, providing the model with the raw information it needs to understand a situation. Each resource has its own URI like `file:///path/to/document.md`. Example resources:
   - calendar defined by URI like `calendar://events/2024` returning events of 2024
   - database records defined by URI `postgres://internal/customer/{id}` which contains variable component

In [ ]:
// Example resources definition
{
  "uriTemplate": "weather://forecast/{city}/{date}",
  "name": "weather-forecast",
  "title": "Weather Forecast",
  "description": "Get weather forecast for any city and date",
  "mimeType": "application/json"
}

{
  "uriTemplate": "travel://flights/{origin}/{destination}",
  "name": "flight-search",
  "title": "Flight Search",
  "description": "Search available flights between cities",
  "mimeType": "application/json"
}

2. **Tools:** are executable functions that the client can invoke. Unlike resources, tools allow AI system to perform actions that change the state of a system or fetch dynamic results based on arguments. These require explicit user approval before execution in most MCP implementations. Example:
    - `create_jira_issue`
    - `run_unit_tests`
    - `calculate_hash`

In [ ]:
// Example tool definition
{
  name: "searchFlights",
  description: "Search for available flights",
  inputSchema: {
    type: "object",
    properties: {
      origin: { type: "string", description: "Departure city" },
      destination: { type: "string", description: "Arrival city" },
      date: { type: "string", format: "date", description: "Travel date" }
    },
    required: ["origin", "destination", "date"]
  }
}

3. **Prompts:** are pre-defined templates or *slash commands* provided by the server. They help guide the AI's behavior or structure its interaction for specific tasks. Instead of the user writing a long instruction, they can trigger a prompt template that has been optimized for that server's data. Example:
    - `review_code` prompt that automatically attaches required files and establishes review standards
    - `analyze-logs` prompt that sets the persona of a SRE and prepares the model to parse specific stack traces

In [ ]:
// Example prompt definition
{
  "name": "plan-vacation",
  "title": "Plan a vacation",
  "description": "Guide through vacation planning process",
  "arguments": [
    { "name": "destination", "type": "string", "required": true },
    { "name": "duration", "type": "number", "description": "days" },
    { "name": "budget", "type": "number", "required": false },
    { "name": "interests", "type": "array", "items": { "type": "string" } }
  ]
}

Primitives offer a set of methods that MCP client can use to access the primitive:
1. **Resources Methods**:
    1. `resources/list`: basically asks server what data is has. The server can respond with one or more URIs like: `postgres://internal/customer/{id}`.
    2.  `resources/read`: give data for specific URL. Example: `resources/read` with `postgres://internal/customer/5` means give data for customer with id 5. The MCP server would internally run the required SQL commmand and respond back with data.
    3.  `resources/subscribe`: A client can ask to be notified when a resource changes.

2. **Tools Methods**:
    1. `tools/list`: client asks for a list of available functions, including their parameters. 
    2.  `tools/call`: client provides the tool name and the necessary arguments. Example: The client calls `tools/call` with
        ```json
        {
            "name": "query_db", 
            "arguments": {
                "sql": "SELECT count(*) FROM transactions WHERE status = 'pending'"
            }
        }
        ```

3. **Prompts Methods:**
    1. `prompts/list`: client asks for available templates. As an example: The server offers a prompt called `explain-query-plan`.
    2. `prompts/get`: client requests the actual text of the prompt, often providing arguments to fill in a template. Example: client calls `prompts/get` with
       ```json
           { 
               "name": "explain-query-plan", 
               "arguments": {
                   "query": "SELECT * FROM users"
               }
           }
        ```
        The server returns a long system instruction telling the model how to interpret PostgreSQL `EXPLAIN ANALYZE` output.

## Implementation
MCP Server implementation with Spring AI exposing tools:

In [ ]:
@SpringBootApplication
public class WeatherMcpApplication {

    public static void main(String[] args) {
        SpringApplication.run(WeatherMcpApplication.class, args);
    }

    // Looks at @Tool annotated methods
    @Bean
    public ToolCallbackProvider weatherTools(WeatherService weatherService) {
        return MethodToolCallbackProvider.builder().toolObjects(weatherService).build();
    }
}

@Service
public class WeatherService {

    private final WeatherClient weatherClient;
    private final ObjectMapper objectMapper;

    public WeatherService() {
        this.weatherClient = new WeatherClient();
        this.objectMapper = new ObjectMapper();
    }

    // Some DTOs defined

    @Tool(description = "Get weather forecast for a specific latitude/longitude")
    public String getWeatherForecastByLocation(double latitude, double longitude) {
        Points points = weatherClient.getPoints(pointsUrl);
        Forecast forecast = weatherClient.getForecast(points.properties().forecast());

        return forecast.properties().periods().stream()
                .map(p -> String.format("""
                        %s:
                        Temperature: %s %s
                        Wind: %s %s
                        Forecast: %s
                        
                        """,
                        p.name(),
                        p.temperature(),
                        p.temperatureUnit(),
                        p.windSpeed(),
                        p.windDirection(),
                        p.detailedForecast()))
                .collect(Collectors.joining());
    }

    @Tool(description = "Get weather alerts for a US state. Input is Two-letter US state code (e.g. CA, NY)")
    public String getAlerts(@ToolParam(description = "Two-letter US state code (e.g. CA, NY)") String state) {
        Alert alert = weatherClient.getAlert(url);

        return alert.features().stream()
                .map(f -> String.format("""
                        Event: %s
                        Area: %s
                        Severity: %s
                        Description: %s
                        Instructions: %s
                        
                        """,
                        f.properties().event(),
                        f.properties().areaDesc(),
                        f.properties().severity(),
                        f.properties().description(),
                        f.properties().instruction()))
                .collect(Collectors.joining("\n"));
    }
}

This can be tested using by executing: `npx @modelcontextprotocol/inspector java -jar weather-mcp-0.0.1-SNAPSHOT.jar`. This provides a UI to test all the exposed tools. Example `tool/call`:

In [ ]:
{
  "method": "tools/call",
  "params": {
    "name": "getAlerts",
    "arguments": {
      "state": "NY"
    },
    "_meta": {
      "progressToken": 0
    }
  }
}